# PPO: learn with a clipped policy objective

Learn **Proximal Policy Optimization (PPO-Clip)** from basic Gymnasium, NumPy, and PyTorch components on LunarLander. Read [TRPO](06_trpo.ipynb) first: we reuse its separate actor and critic, on-policy rollouts, and generalized advantage estimation (GAE). PPO replaces the conjugate-gradient solve and line search with ordinary Adam updates on a clipped objective.

Our objective is the expected discounted return
$$J(\theta)=\mathbb E_{\pi_\theta}\left[\sum_{t=0}^{T-1}\gamma^t r_{t+1}\right],$$
where $\theta$ denotes actor parameters, $\pi_\theta(a\mid s)$ the action distribution, $s_t$ a state, $a_t$ an action, $r_{t+1}$ its reward, $\gamma$ the discount factor, and $T$ the episode length.

Let $\theta_{old}$ be the frozen collecting policy and $\hat A_t$ an estimated advantage. Define the probability ratio and clipped surrogate
$$\rho_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)},\qquad
L^{CLIP}(\theta)=\frac1N\sum_t\min\left(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t\right).$$
Here $N$ is the number of sampled transitions, $\epsilon$ is the clipping range, and $\operatorname{clip}$ clamps its first argument between the two bounds. Maximize this surrogate over several shuffled minibatch passes through each fresh rollout. The old probabilities stay fixed throughout those passes.

This notebook implements the clipped variant from [Schulman et al., Proximal Policy Optimization Algorithms (2017)](https://arxiv.org/abs/1707.06347). Clipping removes incentives for some large probability changes; it does **not** impose a hard ratio or KL constraint, nor guarantee improved return.

## 1. Set up a small CPU experiment

`CLIP_RANGE` is $\epsilon$, `LEARNING_RATE` controls the actor's Adam steps, and `VALUE_LEARNING_RATE` controls the critic. `N_EPOCHS` and `BATCH_SIZE` determine how often each rollout is reused. Sampling the categorical policy supplies exploration; an entropy bonus can encourage it further.

LunarLander has eight observation features and four discrete actions. Install its optional physics dependency with `pip install "gymnasium[box2d]>=1.0,<2"` (install `swig` first if the Box2D build requires it). The 50,000-step run is a short CPU demonstration, not a promise of reliable landings. Increase `TOTAL_TIMESTEPS` for a longer experiment. Training uses no rendering; evaluation renders separately.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical, kl_divergence

SEED = 7
ENV_ID = "LunarLander-v3"
TOTAL_TIMESTEPS = 50_000
N_STEPS = 1024
HIDDEN_SIZE = 32
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
LEARNING_RATE = 3e-4
VALUE_LEARNING_RATE = 1e-3
N_EPOCHS = 10
BATCH_SIZE = 64
ENTROPY_COEF = 0.01
MAX_GRAD_NORM = 0.5
EPS = 1e-8
EVAL_EPISODES = 5

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)
device = torch.device("cpu")

env = gym.make(ENV_ID, render_mode="human")
env.metadata["render_fps"] = 1000

observation_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

## 2. Build separate actor and critic networks

The actor maps $s$ to logits $z_\theta(s)$, with $\pi_\theta(a\mid s)=\operatorname{softmax}(z_\theta(s))_a$. The critic $V_\phi(s)$ estimates discounted return with its own parameters $\phi$. Separate networks keep the actor and critic gradient flows independent. Both now have Adam optimizers. `select_action` samples during training and takes the most probable action during evaluation.

In [ ]:
def make_network(output_dim):
    return nn.Sequential(
        nn.Linear(observation_dim, HIDDEN_SIZE),
        nn.Tanh(),
        nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
        nn.Tanh(),
        nn.Linear(HIDDEN_SIZE, output_dim),
    ).to(device)


actor = make_network(action_dim)
critic = make_network(1)
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=LEARNING_RATE)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=VALUE_LEARNING_RATE)


def action_distribution(observations):
    return Categorical(logits=actor(observations))


@torch.no_grad()
def select_action(observation, deterministic=False):
    states = torch.as_tensor(observation, dtype=torch.float32, device=device).unsqueeze(
        0
    )
    distribution = action_distribution(states)
    action = (
        distribution.probs.argmax(dim=-1) if deterministic else distribution.sample()
    )
    return int(action.item())

## 3. Estimate advantages with two boundary masks

We use generalized advantage estimation (GAE):
$$d_t=r_{t+1}+\gamma(1-\mathrm{terminated}_t)V_\phi(s_{t+1})-V_\phi(s_t),$$
$$\hat A_t=d_t+\gamma\lambda(1-\mathrm{episode\_end}_t)\hat A_{t+1},
\qquad y_t=\hat A_t+V_\phi(s_t).$$
Here $d_t$ is the TD residual, $\lambda$ (`GAE_LAMBDA`) mixes multi-step estimates, and $y_t$ is the critic target. The recursion starts at zero beyond the rollout. A true termination removes the bootstrap; a time-limit truncation retains the value of the final observation. **Both** end the trace so it never crosses an environment reset. An ordinary rollout cutoff still bootstraps.

Compute targets without gradients, then standardize advantages for the actor. Keep unnormalized advantages for critic targets. Gradients must not flow through rewards, old policy quantities, advantages, or bootstrap values.

In [ ]:
@torch.no_grad()
def advantage_targets(rewards, values, next_values, terminated, episode_ends):
    residuals = rewards + GAMMA * (1.0 - terminated) * next_values - values
    advantages = torch.empty_like(rewards)
    running_advantage = torch.zeros((), device=device)
    for t in reversed(range(len(rewards))):
        running_advantage = (
            residuals[t]
            + GAMMA * GAE_LAMBDA * (1.0 - episode_ends[t]) * running_advantage
        )
        advantages[t] = running_advantage
    targets = advantages + values
    normalized = (advantages - advantages.mean()) / (
        advantages.std(unbiased=False) + EPS
    )
    return normalized, targets

## 4. Inspect what clipping does

For one transition, the objective is $\ell(\rho,A)=\min(\rho A,\operatorname{clip}(\rho,1-\epsilon,1+\epsilon)A)$, where $A$ is its fixed advantage and $\rho$ its probability ratio. If $A>0$, the incentive to increase the ratio stops above $1+\epsilon$. If $A<0$, the incentive to decrease the ratio stops below $1-\epsilon$. Changes in the harmful direction are still penalized: the minimum matters, especially for negative advantages.

The plot evaluates both expressions for $A=+1$ and $A=-1$. Clipping is part of the objective; we do not clamp network parameters or the action distribution itself.

In [ ]:
ratios = np.linspace(0.5, 1.5, 201)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for axis, advantage in zip(axes, [1.0, -1.0], strict=True):
    surrogate = ratios * advantage
    clipped = np.clip(ratios, 1 - CLIP_RANGE, 1 + CLIP_RANGE) * advantage
    axis.plot(ratios, surrogate, linestyle="--", label="unclipped")
    axis.plot(ratios, np.minimum(surrogate, clipped), label="PPO objective")
    axis.axvline(1 - CLIP_RANGE, color="gray", linestyle=":")
    axis.axvline(1 + CLIP_RANGE, color="gray", linestyle=":")
    axis.set(
        xlabel="Probability ratio", ylabel="Surrogate", title=f"A = {advantage:+g}"
    )
    axis.legend()
plt.tight_layout()
plt.show()

## 5. Reuse a rollout with shuffled minibatch updates

Freeze old action log probabilities $\log\pi_{\theta_{old}}(a_t\mid s_t)$, advantages, and value targets **before** updating either network. Compute ratios as `exp(new_log_probs - old_log_probs)`. Recomputing the denominator after each optimizer step would lose the reference to the collecting policy.

For each minibatch $B$, minimize the actor loss and critic loss
$$L_\pi=-\frac1{|B|}\sum_{t\in B}\ell(\rho_t,\hat A_t)-c_H\frac1{|B|}\sum_{t\in B}H(\pi_\theta(\cdot\mid s_t)),\qquad
L_V=\frac1{|B|}\sum_{t\in B}(V_\phi(s_t)-y_t)^2.$$
Here $|B|$ is minibatch size, $c_H=$ `ENTROPY_COEF`, and $H(\pi)=-\sum_a\pi(a)\log\pi(a)$ is categorical entropy. The leading minus sign turns surrogate maximization into loss minimization. The critic uses the fixed, unnormalized GAE targets $y_t$ from section 3; only actor advantages are standardized over the full rollout.

`randperm` reshuffles indices each epoch, including a final partial minibatch. Gradients flow through the current actor and critic only. `clip_grad_norm_` rescales a gradient $g$ by $\min(1,m/\|g\|_2)$, where $m=$ `MAX_GRAD_NORM`; this is separate from probability-ratio clipping.

After all epochs, measure full-rollout mean $D_{KL}(\pi_{old}\|\pi_\theta)$, summing over all actions at each state. The clip fraction is the proportion of sampled-action ratios outside $[1-\epsilon,1+\epsilon]$; it is not exactly the fraction with zero surrogate gradient, because advantage signs matter. These are diagnostics, with no KL early stopping or line search in this implementation.

In [ ]:
def update(rollout):
    observations, actions, rewards, next_observations, terminated, ends = zip(
        *rollout, strict=True
    )

    def tensor(data):
        return torch.as_tensor(np.asarray(data), dtype=torch.float32, device=device)

    states, next_states = tensor(observations), tensor(next_observations)
    actions = torch.as_tensor(actions, dtype=torch.int64, device=device)
    with torch.no_grad():
        old_distribution = Categorical(logits=actor(states).detach().clone())
        old_log_probs = old_distribution.log_prob(actions)
        values = critic(states).squeeze(-1)
        next_values = critic(next_states).squeeze(-1)
        advantages, targets = advantage_targets(
            tensor(rewards), values, next_values, tensor(terminated), tensor(ends)
        )

    for _ in range(N_EPOCHS):
        indices = torch.randperm(len(rollout), device=device)
        for start in range(0, len(rollout), BATCH_SIZE):
            batch = indices[start : start + BATCH_SIZE]
            distribution = action_distribution(states[batch])
            ratio = torch.exp(
                distribution.log_prob(actions[batch]) - old_log_probs[batch]
            )
            surrogate = ratio * advantages[batch]
            clipped = ratio.clamp(1 - CLIP_RANGE, 1 + CLIP_RANGE) * advantages[batch]
            actor_loss = (
                -torch.minimum(surrogate, clipped).mean()
                - ENTROPY_COEF * distribution.entropy().mean()
            )
            actor_optimizer.zero_grad()
            actor_loss.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
            actor_optimizer.step()

            value_loss = nn.functional.mse_loss(
                critic(states[batch]).squeeze(-1), targets[batch]
            )
            critic_optimizer.zero_grad()
            value_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), MAX_GRAD_NORM)
            critic_optimizer.step()

    with torch.no_grad():
        distribution = action_distribution(states)
        ratio = torch.exp(distribution.log_prob(actions) - old_log_probs)
        return {
            "kl": kl_divergence(old_distribution, distribution).mean().item(),
            "clip_fraction": ((ratio - 1).abs() > CLIP_RANGE).float().mean().item(),
            "entropy": distribution.entropy().mean().item(),
            "value_loss": nn.functional.mse_loss(
                critic(states).squeeze(-1), targets
            ).item(),
        }

## 6. Collect on-policy rollouts and train

For each transition $(s_t,a_t,r_{t+1},s_{t+1})$, record both termination and episode-boundary flags. Store the final observation **before** resetting. Keep the policy fixed for `N_STEPS` transitions, then update; the final shorter rollout is also consumed. Episode return is the undiscounted sum $R=\sum_t r_{t+1}$. Its accumulator persists across rollout boundaries.
The policy is fixed during collection, so `update` can snapshot its old distribution before the first optimizer step. Reuse only this rollout for `N_EPOCHS`, then discard it and collect fresh on-policy data. There is no replay buffer.

In [ ]:
def train(total_timesteps):
    episode_returns, history, rollout = [], [], []
    episode_return = 0.0
    observation, _ = env.reset(seed=SEED)
    try:
        for step in range(1, total_timesteps + 1):
            action = select_action(observation)
            next_observation, reward, terminated, truncated, _ = env.step(action)
            episode_end = terminated or truncated
            rollout.append(
                (
                    observation.copy(),
                    action,
                    reward,
                    next_observation.copy(),
                    terminated,
                    episode_end,
                )
            )
            episode_return += reward
            if episode_end:
                episode_returns.append(episode_return)
                episode_return = 0.0
                observation, _ = env.reset()
            else:
                observation = next_observation
            if len(rollout) == N_STEPS or step == total_timesteps:
                metrics = update(rollout)
                history.append(metrics)
                rollout.clear()
                last_return = np.mean(episode_returns[-10:]) if episode_returns else 0
                print(
                    f"Step {step:5d} | episodes {len(episode_returns)}"
                    f" | last return {last_return:6.1f}"
                    f" | KL {metrics['kl']:.5f}"
                    f" | clip fraction {metrics['clip_fraction']:.3f}"
                    f" | entropy {metrics['entropy']:.2f}"
                )
    finally:
        env.close()
    return episode_returns, history


episode_returns, history = train(TOTAL_TIMESTEPS)

## 7. Inspect returns and policy-change diagnostics

The moving average $\bar R_i=\tfrac1w\sum_{j=i-w+1}^i R_j$ uses a window $w$ of up to ten episodes, where $R_j$ is episode $j$'s undiscounted return. Compare it with full-rollout KL, clip fraction, critic mean squared error, and mean policy entropy after each update.

Large KL or clip fractions can motivate a smaller learning rate or fewer epochs. Entropy falling toward zero indicates increasingly deterministic action choices. Critic loss can rise as the policy visits new states. None of these signals alone proves learning success; compare returns across seeds and longer runs.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
axes[0].plot(episode_returns, alpha=0.35, label="episode return")
if episode_returns:
    window = min(10, len(episode_returns))
    moving_average = np.convolve(
        episode_returns, np.ones(window) / window, mode="valid"
    )
    axes[0].plot(
        np.arange(window - 1, len(episode_returns)),
        moving_average,
        label=f"{window}-episode mean",
    )
axes[0].set(title=f"PPO on {ENV_ID}", xlabel="Episode", ylabel="Return")
axes[0].legend()
axes[1].plot([item["kl"] for item in history], label="measured KL")
axes[1].set(title="Policy change", xlabel="Update", ylabel="Mean KL")
axes[1].legend()
axes[2].plot([item["clip_fraction"] for item in history])
axes[2].set(title="Ratios outside clip range", xlabel="Update", ylabel="Fraction")
axes[3].plot([item["value_loss"] for item in history])
axes[3].set(title="Critic fit", xlabel="Update", ylabel="Mean squared error")
axes[4].plot([item["entropy"] for item in history])
axes[4].set(title="Policy entropy", xlabel="Update", ylabel="Entropy (nats)")
axes[5].axis("off")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Evaluate the deterministic policy

Choose $a_t=\arg\max_a\pi_\theta(a\mid s_t)$ in a separate rendered environment and report the mean and standard deviation of undiscounted returns. This measures the modal policy, which differs from the stochastic training policy. `rgb_array` rendering displays a final frame inline and works without a desktop window.

Restart the kernel for each comparison so weights and random seeds are reset. Try `CLIP_RANGE` values of 0.1 and 0.3, fewer `N_EPOCHS`, or `ENTROPY_COEF = 0.0`. How do KL, clip fraction, entropy, and returns change? Why can KL grow even with a small clipping range? Compare this first-order update with TRPO's constrained step. Setting `CLIP_RANGE = 0` does not freeze the policy: harmful ratio changes and the entropy bonus can still produce gradients.

In [ ]:
env = gym.make(ENV_ID, render_mode="human")
env.metadata["render_fps"] = 30

evaluation_returns = []
try:
    for episode in range(EVAL_EPISODES):
        observation, _ = env.reset(seed=SEED + 100 + episode)
        episode_return = 0.0
        while True:
            action = select_action(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = env.step(action)
            episode_return += reward
            if terminated or truncated:
                break
        evaluation_returns.append(episode_return)
        print(f"Episode {episode + 1}: return={episode_return:.1f}")
    final_frame = env.render()
finally:
    env.close()

print(
    f"Mean return: {np.mean(evaluation_returns):.1f} "
    f"+/- {np.std(evaluation_returns):.1f}"
)
